In [ ]:
#Datasets
#Hey, there are these three separate files:
#customers.csv
#products.csv
#orders.csv
#Objective
#You are working as a business data analyst. The company stores customer, product, and order
#information in separate datasets. Your task is to combine these datasets and perform a complete business
#analysis. This assignment focuses heavily on merging and combining DataFrames.

In [10]:
#Part A — Understanding Individual Datasets
#1. Load all three CSV files.
import pandas as pd
customer_dataset = pd.read_csv('../data/customers.csv')
product_dataset = pd.read_csv('../data/products.csv')
order_dataset = pd.read_csv('../data/orders.csv')

customers_df = pd.DataFrame(customer_dataset)
products_df = pd.DataFrame(product_dataset)
orders_df = pd.DataFrame(order_dataset)

#2. Display the dimensions of each DataFrame.
print('Dimensions of customer dataframe', customers_df.shape)
print('Dimensions of product dataframe', products_df.shape)
print('Dimensions of order dataframe', orders_df.shape)

#3. Identify the columns in each dataset.
print("Columns in customer dataset", customers_df.info())
print("Columns in product dataset", products_df.info())
print("Columns in order dataset", orders_df.info())

#4. Identify the common columns between datasets.
dfs = [customers_df, products_df, orders_df]
#common_across_all = list(set(dfs[0].columns).intersection(*[df.columns for df in dfs[1:]]))
# 1. Clean up hidden whitespaces from all 3 dataframes
customers_df.columns = customers_df.columns.str.strip().str.lower()
products_df.columns = products_df.columns.str.strip().str.lower()
orders_df.columns = orders_df.columns.str.strip().str.lower()

# 2. Convert columns to sets
set1 = set(customers_df.columns)
set2 = set(products_df.columns)
set3 = set(orders_df.columns)

# 3. Find columns present in ALL three dataframes
common_cols = list(set1.intersection(set2, set3))
print("common coulmns ::", common_cols)

#5. Determine how the three datasets are related.
# 1. Compare Customers and Orders
cust_orders_common = list(set(customers_df.columns).intersection(orders_df.columns))

# 2. Compare Products and Orders
prod_orders_common = list(set(products_df.columns).intersection(orders_df.columns))

# 3. Compare Customers and Products
cust_prod_common = list(set(customers_df.columns).intersection(products_df.columns))

print("--- Overlapping Columns Between Datasets ---")
print(f"• Customers <-> Orders common columns: {cust_orders_common}")
print(f"• Products  <-> Orders common columns: {prod_orders_common}")
print(f"• Customers <-> Products common columns: {cust_prod_common}")

Dimensions of customer dataframe (3000, 5)
Dimensions of product dataframe (100, 4)
Dimensions of order dataframe (15000, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Customer_ID    3000 non-null   object
 1   Customer_Name  3000 non-null   object
 2   City           3000 non-null   object
 3   Age            3000 non-null   int64 
 4   Gender         3000 non-null   object
dtypes: int64(1), object(4)
memory usage: 117.3+ KB
Columns in customer dataset None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Product_ID    100 non-null    object 
 1   Product_Name  100 non-null    object 
 2   Category      100 non-null    object 
 3   Price         100 non-null    float64
dtypes: float64(1), ob

In [11]:
#Part B — Data Cleaning
#6. Check each DataFrame for missing values.
#for _df in dfs:
 #    missing_values = _df.isna()
print("--- 6. Missing Values ---")
print("Customers missing data:\n", customers_df.isnull().sum(), "\n")
print("Products missing data:\n", products_df.isnull().sum(), "\n")
print("Orders missing data:\n", orders_df.isnull().sum(), "\n")

#7. Check for duplicate records.
print("--- 7. Duplicate Records ---")
print(f"Duplicate customer records: {customers_df.duplicated().sum()}")
print(f"Duplicate product records:  {products_df.duplicated().sum()}")
print(f"Duplicate order records:    {orders_df.duplicated().sum()}\n")   

#8. Check the validity of Customer_ID and Product_ID.
print("--- 8. ID Validity ---")
print(f"Is Customer_ID unique in Customers table?: {customers_df['customer_id'].is_unique}")
print(f"Is Product_ID unique in Products table?:   {products_df['product_id'].is_unique}\n")


#9. Convert Order_Date into datetime.
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
print("--- 9. Order_Date Conversion ---")
print(f"Order_Date column data type: {orders_df['order_date'].dtypes}\n")

#10. Check whether every order corresponds to a valid customer.
invalid_customers = orders_df[~orders_df['customer_id'].isin(customers_df['customer_id'])]
print("--- 10. Customer Referential Integrity Check ---")
if invalid_customers.empty:
    print("Success: Every order matches a valid customer.")
else:
     print(f"Warning: Found {len(invalid_customers)} orders with invalid Customer_IDs!")


#11. Check whether every order corresponds to a valid product.
invalid_order = orders_df[~orders_df['product_id'].isin(products_df['product_id'])]
print("\n--- 11. Product Referential Integrity Check ---")
if invalid_order.empty:
    print("Success: Every order matches a valid product.")
else:
     print(f"Warning: Found {len(invalid_order)} orders with invalid Product_IDs!")



--- 6. Missing Values ---
Customers missing data:
 customer_id      0
customer_name    0
city             0
age              0
gender           0
dtype: int64 

Products missing data:
 product_id      0
product_name    0
category        0
price           0
dtype: int64 

Orders missing data:
 order_id       0
customer_id    0
product_id     0
quantity       0
order_date     0
dtype: int64 

--- 7. Duplicate Records ---
Duplicate customer records: 0
Duplicate product records:  0
Duplicate order records:    0

--- 8. ID Validity ---
Is Customer_ID unique in Customers table?: True
Is Product_ID unique in Products table?:   True

--- 9. Order_Date Conversion ---
Order_Date column data type: datetime64[ns]

--- 10. Customer Referential Integrity Check ---
Success: Every order matches a valid customer.

--- 11. Product Referential Integrity Check ---
Success: Every order matches a valid product.


In [15]:
#Part C — Combining Data
#12. Merge orders with customers.
order_cust_df = pd.merge(orders_df, customers_df, on='customer_id', how='inner')
order_cust_df.head(5)

#13. Merge the resulting DataFrame with products.
merged_df = pd.merge(order_cust_df, products_df, on='product_id', how='inner')

#14. Create a final consolidated DataFrame.
merged_df.head(5)

,order_id,customer_id,product_id,quantity,order_date,customer_name,city,age,gender,product_name,category,price
0,O000001,C00861,P0027,4,2025-01-01 00:00:00,Customer_861,Kolkata,51,Female,Product_27,Sports,56171.59
1,O000002,C01295,P0047,3,2025-01-01 02:00:00,Customer_1295,Bangalore,43,Male,Product_47,Clothing,80831.23
2,O000003,C01131,P0099,4,2025-01-01 04:00:00,Customer_1131,Pune,47,Female,Product_99,Electronics,83546.72
3,O000004,C01096,P0006,1,2025-01-01 06:00:00,Customer_1096,Delhi,29,Male,Product_6,Clothing,651.66
4,O000005,C01639,P0060,2,2025-01-01 08:00:00,Customer_1639,Coimbatore,67,Female,Product_60,Sports,81819.68


In [17]:
#Part D — Business Calculations
#15. Create:
#Order_Value = Price × Quantity
merged_df['order_value'] = merged_df['price'] * merged_df['quantity']

#16. Calculate total revenue.
total_revenue = merged_df['order_value'].sum().round(2)
merged_df['total_revenue'] = total_revenue
print("16--Total revenue :", total_revenue )

#17. Calculate average order value.
average_order_value = merged_df['order_value'].mean().round(2)
print("17--Average order value  :", average_order_value )

#18. Find total revenue by product category.
revenue_by_product_category = merged_df.groupby('category')['order_value'].sum().round(2)
print("18--Revenue by product category  :", revenue_by_product_category )

#19. Find total revenue by city.
revenue_by_city = merged_df.groupby('city')['order_value'].sum().round(2)
print("19--Revenue by city  :", revenue_by_city )

#20. Find the top 10 customers by spending.
customers_by_Spending = merged_df.groupby('customer_id')['order_value'].sum()
top_10_customers = customers_by_Spending.sort_values(ascending=False).head(10)
print('----Top 10 customers----\n',top_10_customers)

#21. Find the top 10 products by revenue.
product_by_revenue = merged_df.groupby('product_id')['order_value'].sum()
top_10_products = product_by_revenue.sort_values(ascending=False).head(10)
print('----Top 10 Product----\n', top_10_products)

#22. Find the most frequently purchased product.
most_purchased_product = merged_df['product_id'].mode()[0]
print('----Most Purchased Product----\n', most_purchased_product)

#23. Find the city with the highest number of customers.
# Count unique customers per city
city_counts = merged_df.groupby('city')['customer_id'].nunique()

# Find the city with the highest number of customers
top_city = city_counts.idxmax()
top_count = city_counts.max()

print(f"The city with the highest number of customers is {top_city} with {top_count} customers.")

#24. Find the city generating the highest revenue.
highest_city_revenue = revenue_by_city.idxmax()
print("City with highest revenue ", highest_city_revenue)

16--Total revenue : 2233171871.3
17--Average order value  : 148878.12
18--Revenue by product category  : category
Beauty             2.966918e+08
Books              3.050563e+08
Clothing           4.261489e+08
Electronics        3.312926e+08
Home Appliances    3.313110e+08
Sports             5.426713e+08
Name: order_value, dtype: float64
19--Revenue by city  : city
Bangalore     2.637059e+08
Chennai       2.519685e+08
Coimbatore    2.831225e+08
Delhi         2.654541e+08
Hyderabad     2.932639e+08
Kolkata       3.155500e+08
Mumbai        2.803566e+08
Pune          2.797505e+08
Name: order_value, dtype: float64
----Top 10 customers----
 customer_id
C00019    3176078.87
C02897    2528868.53
C01915    2476932.74
C02709    2400905.35
C00187    2394078.09
C00685    2352464.63
C01297    2345815.39
C00259    2330070.43
C00921    2257528.60
C00271    2226186.64
Name: order_value, dtype: float64
----Top 10 Product----
 product_id
P0074    47258003.68
P0088    47213343.92
P0003    47075130.00
P0